# 05k: Forecast Bike Demand Without Looking Ahead

Build a daily-demand prediction experiment with chronological evaluation and a simple baseline. **Time:** 35–45 minutes. **Resources:** CPU, approximately 1 GB available RAM; the default dataset has 731 rows.

**Objectives:** identify target leakage, separate training/validation/test by date, compare mean absolute error (MAE), and distinguish a reproducible demonstration from evidence about real operations.

**Prerequisites:** complete [the Jupyter introduction](05_Running_Jupyter_Lab.ipynb), select a Python 3.10+ kernel, and follow the [support guide](README.md). Packages: NumPy, pandas, scikit-learn and Matplotlib. Use your approved environment; this notebook does not install packages. CPU only, no GPU, credentials, gateway, or external service required. Run cells in order, then restart the kernel and Run All to check reproducibility.

## Install the notebook dependencies if needed

The next cell installs the shared CPU packages into **this notebook's selected Python kernel** using `%pip`. It includes **scikit-learn**, whose import name is `sklearn`, plus NumPy, pandas, matplotlib, Pillow, and IPython.

If the packages are missing or this is a fresh environment, set `INSTALL_PACKAGES = True` and run the cell once. Only do this when installation is permitted and pip is configured to use your approved package source. If the environment is already prepared, leave it `False`. The cell finds `requirements.txt` whether the kernel starts in the repository root or `05_Jupyter`.

After a successful installation, change the flag back to `False`, use **Kernel → Restart Kernel**, and run the notebook from the top. A restart clears old imports and variables. If installation fails or the source is blocked, stop here and ask the administrator to prepare the environment or approved mirror; do not bypass TLS checks.

PyTorch/CUDA, interactive widgets, ClearML, and live gateway extras remain optional. See the [support guide](README.md#prepare-one-approved-python-environment) before enabling them. Package installation can use the network; the lesson's default dataset path remains offline.


In [ ]:
from pathlib import Path
import sys

# Change to True only when this kernel needs the section's packages installed.
INSTALL_PACKAGES = False

SETUP_HERE = Path.cwd().resolve()
SETUP_LAB = SETUP_HERE if (SETUP_HERE / "image_lab.py").is_file() else SETUP_HERE / "05_Jupyter"
REQUIREMENTS = SETUP_LAB / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise FileNotFoundError("Start this kernel in the repository root or 05_Jupyter folder with requirements.txt present.")
print("Selected kernel Python:", sys.executable)

if INSTALL_PACKAGES:
    from IPython import get_ipython

    shell = get_ipython()
    if shell is None:
        raise RuntimeError("Run this setup cell in the notebook's IPython Python kernel.")
    # Equivalent to %pip install -r "{REQUIREMENTS}"; targets the selected kernel.
    shell.run_line_magic("pip", f'install -r "{REQUIREMENTS}"')
    print("Check pip output for errors. After success, set INSTALL_PACKAGES = False, restart the kernel, and run from the top.")
else:
    print("Installation skipped. If imports fail, set INSTALL_PACKAGES = True and rerun this cell first.")


In [ ]:
from pathlib import Path
import sys
# Accept a kernel launched at the repository root or in this lesson directory.
cwd = Path.cwd().resolve()
LAB_DIR = cwd if (cwd / "05_Running_Jupyter_Lab.ipynb").is_file() else cwd / "05_Jupyter"
if not LAB_DIR.is_dir():
    raise RuntimeError("Start the kernel from the repository root or 05_Jupyter directory.")
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Choose provenance explicitly

The default is **invented, deterministic daily data**, shaped like a bike-sharing table. Its dates and counts are not Capital Bikeshare observations. No claims about actual riders or real forecasting accuracy follow from synthetic results.

Optional real-data mode downloads only UCI's fixed Bike Sharing archive (approximately 300 KB) when you explicitly set `USE_REAL_DATA = True`. The request has a 20-second socket timeout and 1 MB compressed size cap. Only `day.csv` is read in memory, with a 2 MB uncompressed cap; nothing is extracted to disk. If network access is blocked, keep the offline mode. A failing real-data request raises an error rather than silently substituting synthetic observations.

**Real-data attribution:** [UCI Bike Sharing dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), DOI [10.24432/C5W894](https://doi.org/10.24432/C5W894), currently listed by UCI under **CC BY 4.0**. Created by Hadi Fanaee-T; original bike-sharing records from **Capital Bikeshare**, Washington, DC, 2011–2012. Cite Fanaee-T, H. and Gama, J. (2013), *Event labeling combining ensemble detectors and background knowledge*, Progress in Artificial Intelligence. Read the source dataset description and license before reuse. This lesson selects and transforms the daily table; it does not redistribute the source archive.

In [ ]:
USE_REAL_DATA = False  # Explicit opt-in; the default makes no network requests.
SEED = 51

def synthetic_bike_days():
    rng = np.random.default_rng(SEED)
    dates = pd.date_range("2011-01-01", periods=731, freq="D")
    t = np.arange(len(dates))
    annual = np.sin(2 * np.pi * (dates.dayofyear.to_numpy() - 100) / 365.25)
    weekend = (dates.dayofweek >= 5).astype(int)
    counts = np.maximum(0, 2500 + 2.2*t + 1300*annual - 350*weekend + rng.normal(0, 350, len(t)))
    return pd.DataFrame({"dteday": dates, "cnt": counts.round().astype(int)})

def load_uci_daily():
    import io
    import zipfile
    import urllib.request
    url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *args, **kwargs):
            raise RuntimeError("Unexpected download redirect; inspect the official source manually.")
    opener = urllib.request.build_opener(NoRedirect())
    with opener.open(url, timeout=20) as response:
        payload = response.read(1_000_001)
    if len(payload) > 1_000_000:
        raise ValueError("Archive exceeds the 1 MB download limit.")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        candidates = [x for x in archive.infolist() if x.filename == "day.csv"]
        if len(candidates) != 1 or candidates[0].file_size > 2_000_000:
            raise ValueError("Expected one bounded day.csv in the official archive.")
        with archive.open(candidates[0]) as stream:
            csv_bytes = stream.read(2_000_001)
        if len(csv_bytes) > 2_000_000:
            raise ValueError("Daily CSV exceeds its size limit.")
    return pd.read_csv(io.BytesIO(csv_bytes), parse_dates=["dteday"])

days = load_uci_daily() if USE_REAL_DATA else synthetic_bike_days()
DATA_SOURCE = "UCI / Capital Bikeshare observations" if USE_REAL_DATA else "SYNTHETIC educational data"
days = days.sort_values("dteday").reset_index(drop=True)
assert days.dteday.is_unique and days.dteday.notna().all()
assert days.cnt.notna().all() and (days.cnt >= 0).all()
print(DATA_SOURCE, "|", len(days), "days")
days[["dteday", "cnt"]].head()

## 2. Define the prediction contract, then split

Predict a future day's count using **calendar information known ahead of time** only. `cnt` is the target; `casual` and `registered` sum to the target and must never be inputs. Realized weather on a future day is not known at forecast time. This model excludes temperature, humidity, wind and weather codes. A weather extension would need archived forecasts available at each prediction cutoff, not the day's observed weather.

We do not add lag features here, so the full held-out block can be predicted without reading its targets. Training gets the earliest 60%, validation the next 20%, and test the final 20%. No random splitting or shuffled cross-validation. A calendar split can expose a seasonal or growth shift: that is part of the task, not a reason to reshuffle.

In [ ]:
def calendar_features(dates):
    day = dates.dt.dayofyear.to_numpy()
    return pd.DataFrame({
        "elapsed_days": (dates - pd.Timestamp("2011-01-01")).dt.days.to_numpy(),
        "annual_sin": np.sin(2*np.pi*day/365.25),
        "annual_cos": np.cos(2*np.pi*day/365.25),
        "weekend": (dates.dt.dayofweek >= 5).astype(int).to_numpy(),
    }, index=dates.index)

X = calendar_features(days.dteday)
y = days.cnt
n_train, n_valid_end = int(len(days)*0.60), int(len(days)*0.80)
train = slice(0, n_train)
valid = slice(n_train, n_valid_end)
test = slice(n_valid_end, len(days))
assert days.dteday.iloc[n_train-1] < days.dteday.iloc[n_train]
assert days.dteday.iloc[n_valid_end-1] < days.dteday.iloc[n_valid_end]
assert not {"cnt", "casual", "registered"}.intersection(X.columns)
for name, split in [("train", train), ("validation", valid), ("test", test)]:
    dates = days.dteday.iloc[split]
    print(name, len(dates), str(dates.min().date()), "to", str(dates.max().date()))

**Checkpoint:** each split ends before the next begins. Explain why dropping `cnt` alone would not prevent leakage if `registered` remained.

## 3. Select a model using validation only

Scaling and regression fit only on training rows. Try three predeclared regularization strengths; choose by validation MAE. The baseline is a constant training-period mean. We leave the training fit unchanged after selection so the test result evaluates the exact model that won validation. Negative predictions are clipped to zero consistently for both selection and evaluation.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

models, validation_rows = {}, []
baseline_count = float(y.iloc[train].mean())
for alpha in (0.1, 10.0, 100.0):
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(X.iloc[train], y.iloc[train])
    prediction = np.maximum(0, model.predict(X.iloc[valid]))
    models[alpha] = model
    validation_rows.append({"alpha": alpha, "validation_MAE": mean_absolute_error(y.iloc[valid], prediction)})
validation_scores = pd.DataFrame(validation_rows).sort_values(["validation_MAE", "alpha"])
best_alpha = float(validation_scores.iloc[0].alpha)
selected_model = models[best_alpha]
print(validation_scores.to_string(index=False))
print("Baseline validation MAE:", round(mean_absolute_error(y.iloc[valid], np.full(len(y.iloc[valid]), baseline_count)), 2))
print("Selected alpha:", best_alpha)

In [ ]:
# Touch test labels for final evaluation only, after all choices are locked.
test_prediction = np.maximum(0, selected_model.predict(X.iloc[test]))
test_baseline = np.full(len(y.iloc[test]), baseline_count)
metrics = pd.Series({
    "calendar model test MAE": mean_absolute_error(y.iloc[test], test_prediction),
    "training-mean baseline test MAE": mean_absolute_error(y.iloc[test], test_baseline),
})
print(DATA_SOURCE)
print(metrics.round(2).to_string())
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(days.dteday.iloc[test], y.iloc[test], label="Observed / generated count", alpha=0.7)
ax.plot(days.dteday.iloc[test], test_prediction, label="Calendar model")
ax.plot(days.dteday.iloc[test], test_baseline, label="Training mean", linestyle="--")
ax.set(title=f"Held-out days: {DATA_SOURCE}", xlabel="Date", ylabel="Daily rentals")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Interpret, extend, and stop

**Checkpoint:** report both MAEs in rentals/day. A lower error than this weak baseline does not establish production usefulness. This small, old dataset omits capacity constraints, station-level demand, and contemporary changes. Synthetic scores validate code paths, not Capital Bikeshare forecasts.

**Exercise:** on training and validation only, add weekday sine/cosine features and a training-only weekday-average baseline. Predeclare which validation score selects the final model. Once you inspect test results repeatedly, that interval becomes development data; obtain a new untouched future period for an honest final test. For rolling backtests, retrain each fold using only its past.

**Troubleshooting:** a missing import means the selected kernel lacks a prerequisite; use the approved package setup in the support guide. An HTTP/ZIP error in real mode means the optional source could not be validated; return to explicit synthetic mode or resolve network access, never infer real results. Unexpectedly tiny error should trigger a check for target components and future information.

**Save and cleanup:** save the notebook in your confirmed persistent workspace. No dataset or model files were written by this notebook. Clear outputs if you do not want them in Git, shut down the kernel, and stop the hosted session through its dashboard when finished. Saving alone does not stop compute billing; confirm persistence using the introduction's restore workflow.

[Section index](README.md) · [Next: Detect Telemetry Anomalies](05l_Detect_Telemetry_Anomalies.ipynb)